# Phase 5: Results and Comparison

This notebook is the **final comparative analysis** of the project. It aggregates the quantitative results (PSNR, SSIM) produced by all three methods — **FISTA with Wavelet L1** (NB02), **Hybrid Attention U-Net** (NB03), and **DiffPIR** (NB04) — and produces a unified set of plots, tables, and qualitative comparisons.

The goal is not only to rank the methods numerically, but to reason critically about **why** each method behaves the way it does under the four degradation scenarios:

| Scenario | Scale Factor | Noise Level |
|:---:|:---:|:---:|
| 1 (Easy) | s=2 | σ=0.005 |
| 2 (Medium) | s=2 | σ=0.010 |
| 3 (Medium-Hard) | s=4 | σ=0.005 |
| 4 (Hard) | s=4 | σ=0.010 |

All metrics are computed against the **HR Ground Truth** (256×256), and all methods receive **exactly the same degraded inputs**, generated on-the-fly from the shared test set.

## 5.1 Environment Setup and Drive Mount

We mount Google Drive and reconstruct all the paths used in the previous notebooks.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DRIVE_FOLDER = 'CI_Project'
    DRIVE_PATH = os.path.join('/content/drive/MyDrive', BASE_DRIVE_FOLDER)
else:
    DRIVE_PATH = './CI_Project'

DATASET_PATH  = os.path.join(DRIVE_PATH, 'dataset')
WEIGHTS_PATH  = os.path.join(DRIVE_PATH, 'weights')
OUTPUT_FISTA  = os.path.join(DRIVE_PATH, 'outputs', 'fista')
OUTPUT_UNET   = os.path.join(DRIVE_PATH, 'outputs', 'unet')
OUTPUT_DIFF   = os.path.join(DRIVE_PATH, 'outputs', 'diffpir')
OUTPUT_NB05   = os.path.join(DRIVE_PATH, 'outputs', 'comparison')

os.makedirs(OUTPUT_NB05, exist_ok=True)

print(f"Drive path : {DRIVE_PATH}")
print(f"Output NB05: {OUTPUT_NB05}")

## 5.2 Clone / Update IPPy and Install Dependencies

In [ ]:
import subprocess, sys

if not os.path.exists('IPPy'):
    !git clone https://github.com/alessandrocampedelli/IPPy.git
else:
    %cd IPPy
    !git pull
    %cd ..

!pip install -q PyWavelets

sys.path.append(os.path.abspath('IPPy'))

from IPPy.operators import DownScaling
from IPPy.utilities._utilities import gaussian_noise
print("IPPy imported successfully.")

## 5.3 Imports

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob
import csv
import json
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim_fn

# ── Reproducibility ──────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

## 5.4 Configuration

All hyper-parameters and scenario definitions are centralised here so that the notebook can be re-run with a single edit.

In [ ]:
# ── Degradation scenarios ────────────────────────────────────────────────────
ALL_SCENARIOS = [
    (2, 0.005),
    (2, 0.010),
    (4, 0.005),
    (4, 0.010),
]
SCENARIO_LABELS = [
    r's=2, $\sigma$=0.005  (Easy)',
    r's=2, $\sigma$=0.010  (Medium)',
    r's=4, $\sigma$=0.005  (Medium-Hard)',
    r's=4, $\sigma$=0.010  (Hard)',
]

IMG_SHAPE     = (256, 256)
NUM_QUAL_IMGS = 4    # qualitative grid rows
NUM_EVAL_IMGS = 20   # images used for quantitative eval (same as NB03/NB04)

# ── U-Net architecture (must match NB03) ────────────────────────────────────
UNET_MIDDLE_CH   = [32, 64, 128, 256]
UNET_DOWN_LAYERS = ('ResDownBlock', 'ResDownBlock', 'AttentionDownBlock')
UNET_UP_LAYERS   = ('AttentionUpBlock', 'ResUpBlock', 'ResUpBlock')
UNET_N_HEADS     = 4

# ── DiffPIR / DDPM architecture (must match NB04) ───────────────────────────
DIFF_BASE_CH      = 32
DIFF_CHAN_MULTS   = (1, 2, 2, 4)
DIFF_TIME_DIM     = 128
DIFF_ATTN_LEVELS  = (3,)
T_DIFFUSION       = 1000
DDIM_STEPS        = 100
ETA               = 0.0
RHO_SCALE         = 1.0

# ── FISTA parameters (must match NB02) ──────────────────────────────────────
FISTA_LAMBDA  = 0.01
FISTA_WAVELET = 'db4'
FISTA_LEVEL   = 3
FISTA_MAXITER = 200

print("Configuration loaded.")

## 5.5 Dataset Loader

We reuse the same test split (`.pt` tensors, shape `(1, 256, 256)`, range `[0, 1]`) created in NB01. The degraded observations are generated on-the-fly with `DownScaling` + `gaussian_noise`, exactly as in NB02/NB03/NB04, guaranteeing that **all methods operate on the same inputs**.

In [ ]:
class TestDataset(Dataset):
    """Loads HR tensors from the test split and generates degraded pairs on-the-fly."""

    def __init__(self, data_dir: str, scale_factor: int, noise_level: float,
                 max_images: int | None = None):
        files = sorted(glob.glob(os.path.join(data_dir, '*.pt')))
        if max_images is not None:
            files = files[:max_images]
        self.x_true_list = [torch.load(f) for f in files]   # (1, 256, 256)
        self.K           = DownScaling(IMG_SHAPE, downscale_factor=scale_factor, mode='avg')
        self.noise_level = noise_level

    def __len__(self):
        return len(self.x_true_list)

    def __getitem__(self, idx):
        x_true  = self.x_true_list[idx]                    # (1, 256, 256)
        x_batch = x_true.unsqueeze(0)                      # (1, 1, 256, 256)
        y_clean = self.K(x_batch)
        y_noisy = y_clean + gaussian_noise(y_clean, self.noise_level)
        # Bilinear upsampling to 256×256  (used by U-Net and as baseline)
        y_up    = torch.nn.functional.interpolate(
            y_noisy, size=IMG_SHAPE, mode='bilinear', align_corners=False)
        return (
            x_true,          # (1, 256, 256)  HR ground truth
            y_noisy[0],      # (1, H_lr, W_lr) LR observation  (for FISTA / DiffPIR)
            y_up[0],         # (1, 256, 256)  bilinear upsampled  (for U-Net)
        )


# Quick sanity-check on one scenario
_ds = TestDataset(os.path.join(DATASET_PATH, 'test'), 2, 0.005, max_images=4)
x_t, y_lr, y_up = _ds[0]
print(f"x_true shape : {x_t.shape}")
print(f"y_lr   shape : {y_lr.shape}")
print(f"y_up   shape : {y_up.shape}")

## 5.6 Metric Helpers

We use the same `PSNR` and `SSIM` definitions as `IPPy.utilities.metrics` (range `[0, 1]`).

In [ ]:
import math

def compute_psnr(x_pred: torch.Tensor, x_true: torch.Tensor) -> float:
    """PSNR between two tensors in [0, 1]. Shape: (1, H, W) or (1, 1, H, W)."""
    mse = torch.mean((x_pred.float() - x_true.float()) ** 2).item()
    if mse == 0:
        return 100.0
    return -20.0 * math.log10(math.sqrt(mse))


def compute_ssim(x_pred: torch.Tensor, x_true: torch.Tensor) -> float:
    """SSIM between two tensors in [0, 1]. Shape: (1, H, W) or (1, 1, H, W)."""
    p = x_pred.squeeze().detach().cpu().numpy()
    t = x_true.squeeze().detach().cpu().numpy()
    return float(ssim_fn(p, t, data_range=1.0))


print("Metric helpers defined.")

## 5.7 Load Pre-Computed Results

NB02, NB03, and NB04 each saved a CSV file with columns `method, scale_factor, noise_level, image_idx, psnr_db, ssim`. We load and merge them into a single DataFrame.

> **Note:** if the CSV files are not found on Drive, the corresponding evaluation cells (§5.8, §5.9, §5.10) will re-run each method from scratch and populate the DataFrame directly.

In [ ]:
CSV_FISTA  = os.path.join(OUTPUT_FISTA, 'fista_metrics.csv')
CSV_UNET   = os.path.join(OUTPUT_UNET,  'unet_metrics.csv')
CSV_DIFF   = os.path.join(OUTPUT_DIFF,  'diffpir_metrics.csv')

dfs = []
for path, name in [(CSV_FISTA, 'fista'), (CSV_UNET, 'unet'), (CSV_DIFF, 'diffpir')]:
    if os.path.exists(path):
        df_tmp = pd.read_csv(path)
        # Normalise column names (different notebooks may use slightly different headers)
        df_tmp.columns = [c.strip().lower().replace(' ', '_') for c in df_tmp.columns]
        # NB03 uses 'architecture' column; NB04 uses 'method' — normalise
        if 'architecture' in df_tmp.columns and 'method' not in df_tmp.columns:
            df_tmp.rename(columns={'architecture': 'method'}, inplace=True)
            # NB03 saves both resunet and attn_unet — keep only attn_unet
            df_tmp = df_tmp[df_tmp['method'] == 'attn_unet'].copy()
            df_tmp['method'] = 'unet'
        elif 'method' not in df_tmp.columns:
            df_tmp.insert(0, 'method', name)
        dfs.append(df_tmp)
        print(f"  Loaded {path}  ({len(df_tmp)} rows)")
    else:
        print(f"  [!] Not found: {path}  — will compute on-the-fly in §5.8/5.9/5.10")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    # Ensure numeric types
    df_all['scale_factor'] = df_all['scale_factor'].astype(int)
    df_all['noise_level']  = df_all['noise_level'].astype(float)
    df_all['psnr_db']      = df_all['psnr_db'].astype(float)
    df_all['ssim']         = df_all['ssim'].astype(float)
    print(f"\nCombined DataFrame: {len(df_all)} rows")
    print(df_all.head())
else:
    df_all = pd.DataFrame(columns=['method', 'scale_factor', 'noise_level',
                                   'image_idx', 'psnr_db', 'ssim'])
    print("Empty DataFrame — all methods will be evaluated from scratch.")


## 5.8 (Optional) Re-run FISTA Evaluation

This cell runs only if FISTA results are **not already on Drive**. It reproduces the NB02 evaluation pipeline using the same parameters (`λ=0.01`, wavelet `db4`, 3 levels, max 200 iterations).

In [ ]:
RUN_FISTA = not os.path.exists(CSV_FISTA)

if RUN_FISTA:
    import pywt
    print("FISTA CSV not found — running evaluation from scratch...")

    # ── Proximal operator: soft-thresholding in wavelet domain ────────────────
    def prox_wavelet_l1(x: np.ndarray, lam: float,
                        wavelet: str = 'db4', level: int = 3) -> np.ndarray:
        """Apply soft-thresholding in the wavelet domain (2-D, single channel)."""
        coeffs = pywt.wavedec2(x, wavelet, level=level)
        new_coeffs = [coeffs[0]]   # approximation — not thresholded
        for detail_level in coeffs[1:]:
            new_coeffs.append(
                tuple(np.sign(c) * np.maximum(np.abs(c) - lam, 0)
                      for c in detail_level)
            )
        return pywt.waverec2(new_coeffs, wavelet)

    # ── FISTA solver ──────────────────────────────────────────────────────────
    def fista_reconstruct(y_lr: torch.Tensor, K: DownScaling,
                          lam: float, wavelet: str, level: int,
                          maxiter: int) -> torch.Tensor:
        """
        Minimise  0.5‖Kx - y‖² + λ‖Ψx‖₁  via FISTA + Wavelet L1 proximal.

        Args:
            y_lr  : (1, 1, H_lr, W_lr) noisy LR observation.
            K     : DownScaling operator with .T() adjoint.
        Returns:
            x_hat : (1, 1, 256, 256) reconstruction.
        """
        # Lipschitz constant L = ‖K^T K‖  (estimated via power method, 20 iters)
        b = torch.rand(1, 1, *IMG_SHAPE)
        for _ in range(20):
            b = K.T(K(b))
            L_est = torch.norm(b).item()
            b = b / (L_est + 1e-12)
        L = max(L_est, 1e-6)

        # Initialise
        x = K.T(y_lr)                     # warm start: adjoint of observation
        z = x.clone()
        t = 1.0
        tolf, tolx = 1e-5, 1e-5

        for k in range(maxiter):
            x_prev = x.clone()

            # Gradient step on data fidelity
            grad  = K.T(K(z) - y_lr)
            v     = z - (1.0 / L) * grad

            # Proximal step in wavelet domain (per-image)
            v_np  = v[0, 0].detach().cpu().numpy()
            x_np  = prox_wavelet_l1(v_np, lam / L, wavelet, level)
            # Crop / pad to exactly 256×256
            x_np  = x_np[:IMG_SHAPE[0], :IMG_SHAPE[1]]
            x     = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            x     = torch.clamp(x, 0.0, 1.0)

            # Nesterov momentum
            t_new = (1.0 + math.sqrt(1.0 + 4.0 * t * t)) / 2.0
            z     = x + ((t - 1.0) / t_new) * (x - x_prev)
            t     = t_new

            # Stopping criterion
            diff = torch.norm(x - x_prev).item()
            if diff < tolx * (1.0 + torch.norm(x_prev).item()):
                break

        return x  # (1, 1, 256, 256)

    # ── Evaluation loop ───────────────────────────────────────────────────────
    rows_fista = []
    os.makedirs(OUTPUT_FISTA, exist_ok=True)

    for s, sigma in ALL_SCENARIOS:
        ds  = TestDataset(os.path.join(DATASET_PATH, 'test'), s, sigma,
                          max_images=NUM_EVAL_IMGS)
        K_s = DownScaling(IMG_SHAPE, downscale_factor=s, mode='avg')
        print(f"  FISTA  s={s}  σ={sigma}  ({len(ds)} images)")

        for idx in range(len(ds)):
            x_true, y_lr, _ = ds[idx]
            y_lr_b = y_lr.unsqueeze(0)   # (1, 1, H_lr, W_lr)
            x_hat  = fista_reconstruct(y_lr_b, K_s,
                                       FISTA_LAMBDA, FISTA_WAVELET,
                                       FISTA_LEVEL,  FISTA_MAXITER)
            psnr_v = compute_psnr(x_hat[0], x_true)
            ssim_v = compute_ssim(x_hat[0], x_true)
            rows_fista.append(['fista', s, sigma, idx, round(psnr_v, 4), round(ssim_v, 4)])

        psnr_mean = np.mean([r[4] for r in rows_fista
                             if r[1] == s and r[2] == sigma])
        ssim_mean = np.mean([r[5] for r in rows_fista
                             if r[1] == s and r[2] == sigma])
        print(f"    Mean PSNR={psnr_mean:.2f} dB   Mean SSIM={ssim_mean:.4f}")

    with open(CSV_FISTA, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['method', 'scale_factor', 'noise_level',
                         'image_idx', 'psnr_db', 'ssim'])
        writer.writerows(rows_fista)

    df_fista = pd.DataFrame(rows_fista,
                            columns=['method', 'scale_factor', 'noise_level',
                                     'image_idx', 'psnr_db', 'ssim'])
    df_all   = pd.concat([df_all, df_fista], ignore_index=True)
    print(f"FISTA CSV saved to {CSV_FISTA}")

else:
    print("FISTA CSV already loaded — skipping re-evaluation.")

## 5.9 (Optional) Re-run U-Net Evaluation

This cell runs only if U-Net results are **not already on Drive**. It loads the best Hybrid Attention U-Net weights saved in NB03 (one `.pth` file per scenario) and runs inference on the test set.

In [ ]:
RUN_UNET = not os.path.exists(CSV_UNET)

if RUN_UNET:
    from IPPy.nn.models import UNet
    print("U-Net CSV not found — running evaluation from scratch...")

    rows_unet = []
    os.makedirs(OUTPUT_UNET, exist_ok=True)

    for s, sigma in ALL_SCENARIOS:
        scenario_key  = f"attn_unet_s{s}_sig{str(sigma).replace('.', '')}"
        weights_file  = os.path.join(WEIGHTS_PATH, f"{scenario_key}_weights.pth")
        config_file   = None   # NB03 does not save a config.json

        if not os.path.exists(weights_file):
            print(f"  [!] Weights not found: {weights_file} — skipping scenario s={s} σ={sigma}")
            continue

        # Architecture config (NB03 does not save config.json — use known config)
        cfg = dict(
            ch_in=1, ch_out=1,
            middle_ch=UNET_MIDDLE_CH,
            n_layers_per_block=2,
            down_layers=UNET_DOWN_LAYERS,
            up_layers=UNET_UP_LAYERS,
            n_heads=UNET_N_HEADS,
            final_activation='sigmoid',
        )

        model = UNet(**cfg).to(DEVICE)
        model.load_state_dict(
            torch.load(weights_file, map_location=DEVICE, weights_only=True)
        )
        model.eval()

        ds  = TestDataset(os.path.join(DATASET_PATH, 'test'), s, sigma,
                          max_images=NUM_EVAL_IMGS)
        print(f"  UNet  s={s}  σ={sigma}  ({len(ds)} images)")

        with torch.no_grad():
            for idx in range(len(ds)):
                x_true, _, y_up = ds[idx]
                y_up_b = y_up.unsqueeze(0).to(DEVICE)   # (1, 1, 256, 256)
                x_hat  = model(y_up_b).cpu()            # (1, 1, 256, 256)
                psnr_v = compute_psnr(x_hat[0], x_true)
                ssim_v = compute_ssim(x_hat[0], x_true)
                rows_unet.append(['unet', s, sigma, idx,
                                  round(psnr_v, 4), round(ssim_v, 4)])

        psnr_mean = np.mean([r[4] for r in rows_unet
                             if r[1] == s and r[2] == sigma])
        ssim_mean = np.mean([r[5] for r in rows_unet
                             if r[1] == s and r[2] == sigma])
        print(f"    Mean PSNR={psnr_mean:.2f} dB   Mean SSIM={ssim_mean:.4f}")

    if rows_unet:
        with open(CSV_UNET, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['method', 'scale_factor', 'noise_level',
                             'image_idx', 'psnr_db', 'ssim'])
            writer.writerows(rows_unet)
        df_unet = pd.DataFrame(rows_unet,
                               columns=['method', 'scale_factor', 'noise_level',
                                        'image_idx', 'psnr_db', 'ssim'])
        df_all  = pd.concat([df_all, df_unet], ignore_index=True)
        print(f"U-Net CSV saved to {CSV_UNET}")

else:
    print("U-Net CSV already loaded — skipping re-evaluation.")


## 5.10 (Optional) Re-run DiffPIR Evaluation

This cell runs only if DiffPIR results are **not already on Drive**. It loads the single EMA diffusion model saved in NB04 and runs `diffpir_reconstruct` for each of the four scenarios.

In [ ]:
RUN_DIFF = not os.path.exists(CSV_DIFF)

if RUN_DIFF:
    from IPPy.nn.diffusion import DiffusionUNet
    from IPPy.nn.diffpir   import (
        get_sr_psf_fft, hqs_proximal_sr, ddim_update, diffpir_reconstruct
    )
    print("DiffPIR CSV not found — running evaluation from scratch...")

    DIFF_WEIGHTS_FILE = os.path.join(WEIGHTS_PATH, 'DDPMDenoiser_retinal.pth')
    if not os.path.exists(DIFF_WEIGHTS_FILE):
        print(f"  [!] DiffPIR weights not found at {DIFF_WEIGHTS_FILE}")
        print("      Please run NB04 first to train the DDPM.")
        RUN_DIFF = False

if RUN_DIFF:
    diffusion_model = DiffusionUNet(
        in_ch=1,
        base_ch=DIFF_BASE_CH,
        channel_mults=DIFF_CHAN_MULTS,
        time_dim=DIFF_TIME_DIM,
        dropout=0.05,
        attn_levels=DIFF_ATTN_LEVELS,
    ).to(DEVICE)
    diffusion_model.load_state_dict(
        torch.load(DIFF_WEIGHTS_FILE, map_location=DEVICE, weights_only=True)
    )
    diffusion_model.eval()

    # Cosine schedule (must match NB04)
    from IPPy.nn.diffusion import cosine_beta_schedule
    betas       = cosine_beta_schedule(T_DIFFUSION).to(DEVICE)
    alphas      = 1.0 - betas
    alpha_bars  = torch.cumprod(alphas, dim=0)
    ddim_idx    = torch.linspace(0, T_DIFFUSION - 1, DDIM_STEPS, dtype=torch.long)
    ab_ddim     = alpha_bars[ddim_idx]

    rows_diff = []
    os.makedirs(OUTPUT_DIFF, exist_ok=True)

    for s, sigma in ALL_SCENARIOS:
        H_fft = get_sr_psf_fft(s, *IMG_SHAPE, device=DEVICE)
        ds    = TestDataset(os.path.join(DATASET_PATH, 'test'), s, sigma,
                            max_images=NUM_EVAL_IMGS)
        print(f"  DiffPIR  s={s}  σ={sigma}  ({len(ds)} images)")

        for idx in range(len(ds)):
            x_true, y_lr, _ = ds[idx]
            y_lr_b = y_lr.unsqueeze(0).to(DEVICE)
            x_hat  = diffpir_reconstruct(
                model             = diffusion_model,
                y_lr_01           = y_lr_b,
                scale_factor      = s,
                sigma_n           = sigma,
                alpha_bars_ddim   = ab_ddim.to(DEVICE),
                ddim_step_indices = ddim_idx,
                H_fft             = H_fft,
                img_shape         = IMG_SHAPE,
                rho_scale         = RHO_SCALE,
                eta               = ETA,
            ).cpu()
            psnr_v = compute_psnr(x_hat[0], x_true)
            ssim_v = compute_ssim(x_hat[0], x_true)
            rows_diff.append(['diffpir', s, sigma, idx,
                              round(psnr_v, 4), round(ssim_v, 4)])

        psnr_mean = np.mean([r[4] for r in rows_diff
                             if r[1] == s and r[2] == sigma])
        ssim_mean = np.mean([r[5] for r in rows_diff
                             if r[1] == s and r[2] == sigma])
        print(f"    Mean PSNR={psnr_mean:.2f} dB   Mean SSIM={ssim_mean:.4f}")

    with open(CSV_DIFF, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['method', 'scale_factor', 'noise_level',
                         'image_idx', 'psnr_db', 'ssim'])
        writer.writerows(rows_diff)

    df_diff = pd.DataFrame(rows_diff,
                           columns=['method', 'scale_factor', 'noise_level',
                                    'image_idx', 'psnr_db', 'ssim'])
    df_all  = pd.concat([df_all, df_diff], ignore_index=True)
    print(f"DiffPIR CSV saved to {CSV_DIFF}")

else:
    print("DiffPIR CSV already loaded — skipping re-evaluation.")


## 5.11 Summary Statistics Table

We compute **mean ± std** of PSNR and SSIM for every (method, scenario) pair and display the full comparison table.

In [ ]:
# ── Aggregate ──────────────────────────────────────────────────────────────
df_all['scale_factor'] = df_all['scale_factor'].astype(int)
df_all['noise_level']  = df_all['noise_level'].astype(float).round(3)

agg = (
    df_all
    .groupby(['method', 'scale_factor', 'noise_level'])
    .agg(
        PSNR_mean  = ('psnr_db', 'mean'),
        PSNR_std   = ('psnr_db', 'std'),
        SSIM_mean  = ('ssim',    'mean'),
        SSIM_std   = ('ssim',    'std'),
        n_images   = ('psnr_db', 'count'),
    )
    .reset_index()
)

agg['PSNR'] = agg.apply(
    lambda r: f"{r.PSNR_mean:.2f} ± {r.PSNR_std:.2f}", axis=1)
agg['SSIM'] = agg.apply(
    lambda r: f"{r.SSIM_mean:.4f} ± {r.SSIM_std:.4f}", axis=1)

table = agg[['method', 'scale_factor', 'noise_level', 'PSNR', 'SSIM', 'n_images']]
table = table.sort_values(['scale_factor', 'noise_level', 'method'])

print("=" * 85)
print(f"{'Method':<12} {'s':>4} {'σ':>7}  {'PSNR (dB)':>22}  {'SSIM':>22}  {'N':>4}")
print("=" * 85)
for _, row in table.iterrows():
    print(f"{row['method']:<12} {int(row['scale_factor']):>4} {row['noise_level']:>7.3f}"
          f"  {row['PSNR']:>22}  {row['SSIM']:>22}  {int(row['n_images']):>4}")
print("=" * 85)

# Save to CSV for the final report
table.to_csv(os.path.join(OUTPUT_NB05, 'summary_table.csv'), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_NB05, 'summary_table.csv')}")

## 5.12 Quantitative Comparison — Bar Chart (PSNR)

A grouped bar chart with one group per scenario, one bar per method. Error bars show ± 1 standard deviation over the test images.

In [ ]:
methods       = ['fista', 'unet', 'diffpir']
method_labels = ['FISTA + Wavelet L1', 'Hybrid Attention U-Net', 'DiffPIR']
method_colors = ['#4C72B0', '#DD8452', '#55A868']

n_scenarios = len(ALL_SCENARIOS)
n_methods   = len(methods)
x           = np.arange(n_scenarios)
width       = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (metric_col, ylabel, title) in enumerate([
    ('PSNR_mean', 'PSNR (dB)', 'Mean PSNR per Scenario'),
    ('SSIM_mean', 'SSIM',      'Mean SSIM per Scenario'),
]):
    ax  = axes[ax_idx]
    err = 'PSNR_std' if ax_idx == 0 else 'SSIM_std'

    for m_idx, (method, label, color) in enumerate(
            zip(methods, method_labels, method_colors)):
        vals, errs = [], []
        for s, sigma in ALL_SCENARIOS:
            sub = agg[(agg['method'] == method) &
                      (agg['scale_factor'] == s) &
                      (agg['noise_level'].round(3) == round(sigma, 3))]
            if len(sub) == 1:
                vals.append(sub[metric_col].values[0])
                errs.append(sub[err].values[0])
            else:
                vals.append(0.0)
                errs.append(0.0)

        offset = (m_idx - 1) * width
        bars = ax.bar(x + offset, vals, width,
                      label=label, color=color, alpha=0.85,
                      yerr=errs, capsize=4, error_kw=dict(elinewidth=1.2))

        # Value annotations
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + (0.3 if ax_idx == 0 else 0.005),
                        f"{v:.1f}" if ax_idx == 0 else f"{v:.3f}",
                        ha='center', va='bottom', fontsize=7.5, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(SCENARIO_LABELS, fontsize=9, rotation=12, ha='right')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

plt.suptitle('Quantitative Comparison — All Methods, All Scenarios',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'comparison_bar_chart.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.13 Quantitative Comparison — Line Plot (PSNR vs. Scenario)

A line chart makes it easier to appreciate the *degradation trajectory* of each method as the inverse problem becomes harder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax_idx, (metric_col, std_col, ylabel, title) in enumerate([
    ('PSNR_mean', 'PSNR_std', 'PSNR (dB)', 'PSNR Degradation Curve'),
    ('SSIM_mean', 'SSIM_std', 'SSIM',      'SSIM Degradation Curve'),
]):
    ax = axes[ax_idx]
    markers = ['o', 's', '^']

    for m_idx, (method, label, color, marker) in enumerate(
            zip(methods, method_labels, method_colors, markers)):
        vals, errs = [], []
        for s, sigma in ALL_SCENARIOS:
            sub = agg[(agg['method'] == method) &
                      (agg['scale_factor'] == s) &
                      (agg['noise_level'].round(3) == round(sigma, 3))]
            if len(sub) == 1:
                vals.append(sub[metric_col].values[0])
                errs.append(sub[std_col].values[0])
            else:
                vals.append(np.nan)
                errs.append(0.0)

        vals = np.array(vals, dtype=float)
        errs = np.array(errs, dtype=float)
        ax.plot(range(n_scenarios), vals,
                marker=marker, color=color, label=label,
                linewidth=2, markersize=8)
        ax.fill_between(range(n_scenarios),
                        vals - errs, vals + errs,
                        color=color, alpha=0.15)

    ax.set_xticks(range(n_scenarios))
    ax.set_xticklabels(SCENARIO_LABELS, fontsize=9, rotation=12, ha='right')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(linestyle='--', alpha=0.5)

plt.suptitle('Performance Degradation Across Scenarios',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'comparison_line_plot.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.14 Qualitative Comparison — Visual Grid

For each of the four scenarios we display a row containing:

| Col 1 | Col 2 | Col 3 | Col 4 | Col 5 |
|:---:|:---:|:---:|:---:|:---:|
| Ground Truth | Degraded (LR up-sampled) | FISTA recon. | U-Net recon. | DiffPIR recon. |

The image used is **index 0** of the test set, the same one shown in the per-method notebooks.

In [ ]:
# ── Helper: run all three methods on a single image ────────────────────────
@torch.no_grad()
def reconstruct_all(x_true: torch.Tensor,
                    y_lr:   torch.Tensor,
                    y_up:   torch.Tensor,
                    s: int, sigma: float):
    """
    Returns dict with keys 'fista', 'unet', 'diffpir',
    each a (1, 256, 256) tensor in [0, 1].
    Also returns the bilinear baseline 'bicubic'.
    """
    results = {'bicubic': y_up.clone()}

    # ── FISTA ────────────────────────────────────────────────────────────────
    try:
        import pywt  # already imported if RUN_FISTA was True
        K_s     = DownScaling(IMG_SHAPE, downscale_factor=s, mode='avg')
        y_lr_b  = y_lr.unsqueeze(0)
        x_fista = fista_reconstruct(y_lr_b, K_s,
                                    FISTA_LAMBDA, FISTA_WAVELET,
                                    FISTA_LEVEL,  FISTA_MAXITER)
        results['fista'] = x_fista[0]
    except Exception as e:
        print(f"  [FISTA] {e}")
        results['fista'] = None

    # ── U-Net ────────────────────────────────────────────────────────────────
    try:
        scenario_key = f"attn_unet_s{s}_sig{str(sigma).replace('.', '')}"
        w_file       = os.path.join(WEIGHTS_PATH, f"{scenario_key}_weights.pth")
        from IPPy.nn.models import UNet
        cfg = dict(ch_in=1, ch_out=1, middle_ch=UNET_MIDDLE_CH,
                   n_layers_per_block=2,
                   down_layers=UNET_DOWN_LAYERS, up_layers=UNET_UP_LAYERS,
                   n_heads=UNET_N_HEADS, final_activation='sigmoid')
        net = UNet(**cfg).to(DEVICE)
        net.load_state_dict(
            torch.load(w_file, map_location=DEVICE, weights_only=True))
        net.eval()
        y_up_b         = y_up.unsqueeze(0).to(DEVICE)
        results['unet']= net(y_up_b)[0].cpu()
    except Exception as e:
        print(f"  [UNet] {e}")
        results['unet'] = None

    # ── DiffPIR ───────────────────────────────────────────────────────────────
    try:
        from IPPy.nn.diffpir import get_sr_psf_fft, diffpir_reconstruct
        H_fft  = get_sr_psf_fft(s, *IMG_SHAPE, device=DEVICE)
        y_lr_b = y_lr.unsqueeze(0).to(DEVICE)
        x_diff = diffpir_reconstruct(
            model             = diffusion_model,
            y_lr_01           = y_lr_b,
            scale_factor      = s,
            sigma_n           = sigma,
            alpha_bars_ddim   = ab_ddim.to(DEVICE),
            ddim_step_indices = ddim_idx,
            H_fft             = H_fft,
            img_shape         = IMG_SHAPE,
            rho_scale         = RHO_SCALE,
            eta               = ETA,
        ).cpu()
        results['diffpir'] = x_diff[0]
    except Exception as e:
        print(f"  [DiffPIR] {e}")
        results['diffpir'] = None

    return results


print("reconstruct_all() helper defined.")


In [ ]:
# ── Build the qualitative grid ────────────────────────────────────────────
COL_TITLES = ['Ground Truth', 'Degraded (Bicubic Up)',
              'FISTA + Wavelet L1', 'Hybrid Attention U-Net', 'DiffPIR']
COL_KEYS   = ['gt',      'bicubic', 'fista', 'unet', 'diffpir']
N_COLS     = len(COL_TITLES)
N_ROWS     = len(ALL_SCENARIOS)

fig, axes = plt.subplots(N_ROWS, N_COLS,
                         figsize=(N_COLS * 3.5, N_ROWS * 3.5))

for row_idx, (s, sigma) in enumerate(ALL_SCENARIOS):
    ds           = TestDataset(os.path.join(DATASET_PATH, 'test'),
                               s, sigma, max_images=1)
    x_true, y_lr, y_up = ds[0]

    recons = reconstruct_all(x_true, y_lr, y_up, s, sigma)
    recons['gt'] = x_true

    for col_idx, (col_key, col_title) in enumerate(zip(COL_KEYS, COL_TITLES)):
        ax  = axes[row_idx, col_idx]
        img = recons.get(col_key)

        if img is not None:
            ax.imshow(img.squeeze().detach().cpu().numpy(), cmap='gray',
                      vmin=0, vmax=1)
            if col_key not in ('gt', 'bicubic'):
                psnr_v = compute_psnr(img, x_true)
                ssim_v = compute_ssim(img, x_true)
                ax.set_xlabel(f"PSNR={psnr_v:.2f} dB\nSSIM={ssim_v:.4f}",
                              fontsize=8)
            elif col_key == 'bicubic':
                psnr_v = compute_psnr(y_up, x_true)
                ssim_v = compute_ssim(y_up, x_true)
                ax.set_xlabel(f"PSNR={psnr_v:.2f} dB\nSSIM={ssim_v:.4f}",
                              fontsize=8)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='gray')

        ax.axis('off')

        if row_idx == 0:
            ax.set_title(col_title, fontsize=10, fontweight='bold', pad=6)
        if col_idx == 0:
            ax.set_ylabel(SCENARIO_LABELS[row_idx], fontsize=9,
                          labelpad=6, rotation=90, va='center')
            ax.yaxis.set_label_position('left')
            ax.axis('on')
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_xticks([])
            ax.set_yticks([])

plt.suptitle('Qualitative Comparison — All Methods, All Scenarios',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'qualitative_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.15 Zoomed Detail Comparison (Vascular Structures)

We crop a 64×64 region centred on the optic disc to magnify the high-frequency details (capillary vessels) where the methods differ most visibly.

In [ ]:
# Crop parameters — centred roughly on the optic disc area
CROP_Y, CROP_X   = 100, 140
CROP_SIZE        = 64

def crop(t: torch.Tensor) -> np.ndarray:
    """Returns a (CROP_SIZE, CROP_SIZE) numpy array."""
    arr = t.squeeze().detach().cpu().numpy()
    return arr[CROP_Y:CROP_Y + CROP_SIZE, CROP_X:CROP_X + CROP_SIZE]


fig, axes = plt.subplots(N_ROWS, N_COLS,
                         figsize=(N_COLS * 2.8, N_ROWS * 2.8))

for row_idx, (s, sigma) in enumerate(ALL_SCENARIOS):
    ds            = TestDataset(os.path.join(DATASET_PATH, 'test'),
                                s, sigma, max_images=1)
    x_true, y_lr, y_up = ds[0]
    recons        = reconstruct_all(x_true, y_lr, y_up, s, sigma)
    recons['gt']  = x_true

    for col_idx, (col_key, col_title) in enumerate(zip(COL_KEYS, COL_TITLES)):
        ax  = axes[row_idx, col_idx]
        img = recons.get(col_key)

        if img is not None:
            ax.imshow(crop(img), cmap='gray', vmin=0, vmax=1,
                      interpolation='nearest')
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12, color='gray')

        ax.axis('off')
        if row_idx == 0:
            ax.set_title(col_title, fontsize=9, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(SCENARIO_LABELS[row_idx], fontsize=8,
                          labelpad=4, rotation=90, va='center')
            ax.yaxis.set_label_position('left')
            ax.axis('on')
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_xticks([])
            ax.set_yticks([])

plt.suptitle('Zoomed Detail (64×64 crop around optic disc) — All Methods',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'qualitative_zoom.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.16 PSNR Distribution — Box Plot

Box plots show the **spread** of per-image PSNR values, complementing the mean ± std table. A wide interquartile range signals inconsistent behaviour across test images.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax_idx, (s, sigma) in enumerate(ALL_SCENARIOS):
    ax   = axes[ax_idx]
    data = []
    tick_labels = []

    for method, label in zip(methods, method_labels):
        sub = df_all[
            (df_all['method']       == method) &
            (df_all['scale_factor'] == s) &
            (df_all['noise_level'].round(3) == round(sigma, 3))
        ]['psnr_db'].dropna().tolist()
        data.append(sub)
        tick_labels.append(label)

    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.4),
                    capprops=dict(linewidth=1.4),
                    flierprops=dict(marker='o', markersize=4, alpha=0.5))

    for patch, color in zip(bp['boxes'], method_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticks(range(1, n_methods + 1))
    ax.set_xticklabels(tick_labels, fontsize=9, rotation=10, ha='right')
    ax.set_ylabel('PSNR (dB)', fontsize=10)
    ax.set_title(SCENARIO_LABELS[ax_idx], fontsize=11, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

plt.suptitle('PSNR Distribution per Method and Scenario',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'psnr_boxplot.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.17 PSNR vs. SSIM Scatter Plot

Each point is one (method, scenario) pair. This plot reveals the **Perception-Distortion trade-off**: methods with high PSNR tend to have lower perceptual quality (SSIM), and vice versa.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

scenario_markers = ['o', 's', '^', 'D']

for m_idx, (method, label, color) in enumerate(
        zip(methods, method_labels, method_colors)):
    for sc_idx, (s, sigma) in enumerate(ALL_SCENARIOS):
        sub = agg[
            (agg['method']       == method) &
            (agg['scale_factor'] == s) &
            (agg['noise_level'].round(3) == round(sigma, 3))
        ]
        if len(sub) == 1:
            ax.scatter(
                sub['PSNR_mean'].values[0],
                sub['SSIM_mean'].values[0],
                color=color, marker=scenario_markers[sc_idx],
                s=90, zorder=5,
                label=f"{label} — {SCENARIO_LABELS[sc_idx]}" if m_idx == 0 else None,
            )
            ax.annotate(
                f"s={s},σ={sigma}",
                xy=(sub['PSNR_mean'].values[0], sub['SSIM_mean'].values[0]),
                xytext=(4, 3), textcoords='offset points',
                fontsize=7, color=color, alpha=0.85,
            )

# Method legend patches
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=l)
                  for c, l in zip(method_colors, method_labels)]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')

ax.set_xlabel('Mean PSNR (dB)', fontsize=11)
ax.set_ylabel('Mean SSIM', fontsize=11)
ax.set_title('PSNR vs. SSIM — Perception-Distortion Trade-off',
             fontsize=13, fontweight='bold')
ax.grid(linestyle='--', alpha=0.5)

plt.tight_layout()
save_path = os.path.join(OUTPUT_NB05, 'psnr_ssim_scatter.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.18 Critical Discussion

### 5.18.1 Quantitative Summary

| Rank | Method | Best scenario | Avg PSNR | Key strength |
|:---:|:---|:---:|:---:|:---|
| 🥇 | **Hybrid Attention U-Net** | s=2, σ=0.005 (≈43.7 dB) | ~39.8 dB | Highest PSNR — MSE-optimised |
| 🥈 | **FISTA + Wavelet L1** | s=2 (≈36.1 dB) | ~33.4 dB | No training, guaranteed convergence |
| 🥉 | **DiffPIR** (100-epoch prior) | s=2, σ=0.01 (≈15.5 dB) | ~13.2 dB | Best framework — under-trained prior |

### 5.18.2 Why FISTA Cannot Compete with U-Net at s=4

With a downscaling factor of 4, **93.75% of the spatial information is irreversibly lost** (the null-space of `K` is enormous). The Wavelet L1 prior can only select the *sparsest* solution consistent with the data — but when so little data is available, all plausible solutions are smooth and blurry. The U-Net, having seen thousands of retinal images during training, has learned a much stronger **data-driven prior**: it literally "hallucinates" statistically plausible capillary structures.

### 5.18.3 The Noise-as-Regularisation Paradox in U-Net (NB03)

Counterintuitively, the Hybrid Attention U-Net trained on σ=0.01 often **outperforms** the one trained on σ=0.005. This is a textbook example of **implicit regularisation via noise injection**: the higher noise perturbs the loss landscape, preventing the network from settling into sharp local minima and pushing it towards broader, more generalisable solutions. It also explains why ResUNet at (s=2, σ=0.005) triggered early stopping at epoch 8 — without the smoothing effect of noise, the gradient landscape was too irregular.

### 5.18.4 DiffPIR — Why the Prior Is Weak and How to Fix It

Our DDPM was trained for only **100 epochs on 3329 images with a 6M-parameter architecture**. The characteristic checker-board artefacts visible in the qualitative results are a direct symptom of an insufficiently trained denoiser: when DiffPIR asks the prior at the HQS step "does this configuration look like a retina?", the answer is still largely noise-dominated. The mathematical HQS backbone is correct — the issue is purely the quality of the plugged-in denoiser.

With a properly trained prior (500+ epochs, 100M+ parameters, or fine-tuned from a public medical-imaging diffusion model), DiffPIR would be expected to:
- **Match or exceed U-Net in SSIM** — generative priors synthesise texturally realistic vessels without the MSE-induced blurring.
- **Retain full data consistency** — unlike U-Net, DiffPIR guarantees `‖K x̂ - y‖² ≈ 0` at every step via the HQS proximal operator.
- **Solve all four scenarios with a single model** — the Plug-and-Play separation means zero retraining is needed when changing `s` or `σ`.

### 5.18.5 The Perception-Distortion Trade-off

The scatter plot in §5.17 illustrates the fundamental **Perception-Distortion trade-off** (Blau & Michaeli, 2018): optimising for PSNR (a pixel-wise distortion measure) and optimising for perceptual quality (captured imperfectly by SSIM) are *competing objectives*. The U-Net wins on PSNR because it was trained with MSE loss, which is mathematically equivalent to maximising PSNR. DiffPIR, with a better prior, would move points towards the upper-right corner of the scatter plot — higher SSIM at the cost of slightly lower PSNR.

## 5.19 Final Summary Figure

A single publication-ready figure combining (a) the mean PSNR bar chart and (b) the mean SSIM bar chart, suitable for inclusion in the oral presentation slides.

In [ ]:
fig = plt.figure(figsize=(18, 7))
gs  = gridspec.GridSpec(1, 3, width_ratios=[2, 2, 1.4], wspace=0.35)

ax_psnr  = fig.add_subplot(gs[0])
ax_ssim  = fig.add_subplot(gs[1])
ax_table = fig.add_subplot(gs[2])

# ── PSNR bar ────────────────────────────────────────────────────────────────
for m_idx, (method, label, color) in enumerate(
        zip(methods, method_labels, method_colors)):
    vals, errs = [], []
    for s, sigma in ALL_SCENARIOS:
        sub = agg[(agg['method'] == method) &
                  (agg['scale_factor'] == s) &
                  (agg['noise_level'].round(3) == round(sigma, 3))]
        vals.append(sub['PSNR_mean'].values[0] if len(sub) == 1 else 0)
        errs.append(sub['PSNR_std'].values[0]  if len(sub) == 1 else 0)

    offset = (m_idx - 1) * width
    ax_psnr.bar(x + offset, vals, width,
                label=label, color=color, alpha=0.85,
                yerr=errs, capsize=3)

ax_psnr.set_xticks(x)
ax_psnr.set_xticklabels(SCENARIO_LABELS, fontsize=8, rotation=14, ha='right')
ax_psnr.set_ylabel('Mean PSNR (dB)', fontsize=10)
ax_psnr.set_title('(a) PSNR', fontsize=12, fontweight='bold')
ax_psnr.legend(fontsize=8)
ax_psnr.grid(axis='y', linestyle='--', alpha=0.4)

# ── SSIM bar ────────────────────────────────────────────────────────────────
for m_idx, (method, label, color) in enumerate(
        zip(methods, method_labels, method_colors)):
    vals, errs = [], []
    for s, sigma in ALL_SCENARIOS:
        sub = agg[(agg['method'] == method) &
                  (agg['scale_factor'] == s) &
                  (agg['noise_level'].round(3) == round(sigma, 3))]
        vals.append(sub['SSIM_mean'].values[0] if len(sub) == 1 else 0)
        errs.append(sub['SSIM_std'].values[0]  if len(sub) == 1 else 0)

    offset = (m_idx - 1) * width
    ax_ssim.bar(x + offset, vals, width,
                label=label, color=color, alpha=0.85,
                yerr=errs, capsize=3)

ax_ssim.set_xticks(x)
ax_ssim.set_xticklabels(SCENARIO_LABELS, fontsize=8, rotation=14, ha='right')
ax_ssim.set_ylabel('Mean SSIM', fontsize=10)
ax_ssim.set_title('(b) SSIM', fontsize=12, fontweight='bold')
ax_ssim.legend(fontsize=8)
ax_ssim.grid(axis='y', linestyle='--', alpha=0.4)

# ── Compact summary table ────────────────────────────────────────────────────
ax_table.axis('off')
table_data  = [['Method', 'Best PSNR', 'Best SSIM']]
for method, label in zip(methods, method_labels):
    sub = agg[agg['method'] == method]
    if len(sub):
        best_psnr = f"{sub['PSNR_mean'].max():.2f} dB"
        best_ssim = f"{sub['SSIM_mean'].max():.4f}"
    else:
        best_psnr = best_ssim = 'N/A'
    table_data.append([label.replace(' ', '\n'), best_psnr, best_ssim])

tbl = ax_table.table(
    cellText=table_data[1:], colLabels=table_data[0],
    loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 2.0)
ax_table.set_title('(c) Best values\nacross scenarios',
                   fontsize=10, fontweight='bold')

plt.suptitle(
    'Super-Resolution on Egyptian Medical Retinal Images\n'
    'FISTA + Wavelet L1  vs.  Hybrid Attention U-Net  vs.  DiffPIR',
    fontsize=13, fontweight='bold')

save_path = os.path.join(OUTPUT_NB05, 'final_summary_figure.png')
plt.savefig(save_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

## 5.20 Save All Outputs

We collect all figures and the summary CSV into a single listing for easy reference during the oral presentation.

In [ ]:
print("=" * 60)
print("NB05 — All saved outputs")
print("=" * 60)

expected_outputs = [
    'summary_table.csv',
    'comparison_bar_chart.png',
    'comparison_line_plot.png',
    'qualitative_comparison.png',
    'qualitative_zoom.png',
    'psnr_boxplot.png',
    'psnr_ssim_scatter.png',
    'final_summary_figure.png',
]

for fname in expected_outputs:
    full_path = os.path.join(OUTPUT_NB05, fname)
    status    = '✓' if os.path.exists(full_path) else '✗ MISSING'
    print(f"  [{status}]  {full_path}")

print("=" * 60)
print("NB05 complete — project comparison finished.")